# 🎨 Cartoonify — Unified Interface

Combines **Reimagine** (FLUX.1-Kontext-dev), **Scene** (Depth ControlNet), and **Portrait**
(Canny ControlNet) into a single interface. One story. One button. Three rendering styles.

**Flow:** Write your story → upload a photo → pick a rendering style → click Cartoonify.
Gemini builds the structured prompt silently on generate — no separate Build Prompt step.

**Settings modes:**
- **Default** — hardcoded parameters per rendering style
- **Wild** — Gemini reads the story and suggests parameters optimised for maximum satirical impact

| Style | Pipeline | Best for |
|---|---|---|
| **Reimagine** | FLUX.1-Kontext-dev | Creative reinterpretation — scene can change freely |
| **Scene** | FLUX.1-dev + ControlNet Depth | Crowds, architecture — spatial layout preserved |
| **Portrait** | FLUX.1-dev + ControlNet Canny | Specific person must be immediately recognisable |

**VRAM note:** Only one pipeline loads at a time. Switching modes takes ~60–90 s (one-time per switch).

**Requires:** Google Colab Pro → Runtime → Change runtime type → **A100 GPU**

> Accept the [FLUX.1-Kontext-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev)
> on HuggingFace before running — it is gated separately from FLUX.1-dev.

---

## 1 — Confirm A100 GPU

In [ ]:
!nvidia-smi

## 2 — Install Dependencies

In [ ]:
!pip install --quiet diffusers transformers accelerate sentencepiece
!pip install --quiet gradio
!pip install --quiet huggingface_hub
!pip install --quiet google-genai
!pip install --quiet opencv-python-headless

In [ ]:
import os
os.kill(os.getpid(), 9)
print("IGNORE: session crashed. This is intentional — continue to the next cell.")

## 3 — Imports

In [ ]:
import gc
import json
import queue
import threading
import datetime
import os
import numpy as np
import cv2
import torch
import gradio as gr
import google.genai as genai
import google.genai.types as genai_types
from PIL import Image
from diffusers import (
    FluxPipeline,
    FluxKontextPipeline,
    FluxControlNetPipeline,
    FluxControlNetModel,
)
from transformers import pipeline as hf_pipeline

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'cv2    : {cv2.__version__}')
print(f'gradio : {gr.__version__}')
print(f'genai  : {genai.__version__}')

## 4 — Mount Google Drive

In [ ]:
import shutil
from google.colab import drive

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')

## 5 — API Tokens

**Hugging Face** — FLUX.1-dev and FLUX.1-Kontext-dev are both gated models.
Add a Read token to Colab Secrets as `HF_TOKEN`.

**Google Gemini** — Add your key to Colab Secrets as `GOOGLE_API_KEY`
([aistudio.google.com/apikey](https://aistudio.google.com/apikey)).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN       = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

login(token=HF_TOKEN, add_to_git_credential=False)
print('✓ Logged in to Hugging Face')
print('✓ Google API key loaded' if GOOGLE_API_KEY else '⚠️  GOOGLE_API_KEY not found — story-to-prompt and Wild mode will not work')

## 6 — Configuration

Edit this cell to change the LoRA, trigger word, default rendering mode, or default settings mode.
Nothing else needs to change.

In [ ]:
# ── LoRA ───────────────────────────────────────────────────────────────────────────────
LORA_SOURCE     = 'drive'   # 'huggingface' | 'drive'
LORA_HF_REPO    = 'strangerzonehf/Ghibli-Flux-Cartoon-LoRA'
LORA_HF_WEIGHT  = 'Ghibili-Cartoon-Art.safetensors'
LORA_DRIVE_PATH         = '/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/gdo_cartoon.safetensors'
LORA_GADO_ADAPTER_NAME  = 'gado_cartoon'   # adapter name passed to load_lora_weights / set_adapters

# ── Satirefic LoRA ────────────────────────────────────────────────────────────────
LORA_SATIREFIC_PATH         = '/content/drive/MyDrive/satirefic/satirefic/satirefic.safetensors'
LORA_SATIREFIC_ADAPTER_NAME = 'satirefic'

# ── Style / LoRA selector options ────────────────────────────────────────────────────
# Maps UI label -> adapter name and trigger word used in prompt building.
LORA_STYLES = {
    'None':           {'adapter': None,           'trigger': ''},
    'Gado Cartoon':   {'adapter': 'gado_cartoon',  'trigger': 'gdo_cartoon'},
    'Satirefic LoRA': {'adapter': 'satirefic',     'trigger': 'satirefic'},
}

# ── Trigger + fallback prompt ────────────────────────────────────────────────────
DEFAULT_TRIGGER = 'gdo_cartoon'
DEFAULT_PROMPT  = 'satirical cartoon illustration, bold outlines, vivid flat colours, expressive exaggerated features'

# Global prompt prefix applied to every generation (all modes, all LoRAs, manual or Gemini)
GLOBAL_PROMPT_PREFIX  = 'crude black and white newspaper caricature'

# Satirefic hidden style boost applied only when Satirefic LoRA is selected
SATIREFIC_STYLE_BOOST = (
    'maximum grotesque political satire, extreme caricature exaggeration, '
    'absurd anti-authoritarian newspaper cartoon, rough dirty ink linework, '
    'ugly distorted faces, pig-politician symbolism, visual irony, '
    'anarchist press style, flat white paper, brutal black ink, no readable text'
)

# ── Gemini ──────────────────────────────────────────────────────────────────────────
GOOGLE_MODEL = 'gemini-2.5-flash-lite'

# ── Base models ────────────────────────────────────────────────────────────────────
BASE_MODEL_KONTEXT = 'black-forest-labs/FLUX.1-Kontext-dev'
BASE_MODEL_FLUX    = 'black-forest-labs/FLUX.1-dev'
CONTROLNET_MODEL   = 'Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0'
DEPTH_MODEL        = 'depth-anything/Depth-Anything-V2-Small-hf'

# ── Default parameters per rendering mode ────────────────────────────────────────
DEFAULTS = {
    'imagine':   {'guidance': 3.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'reimagine': {'guidance': 2.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'scene':     {'guidance': 3.5, 'cn_scale': 0.8, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
    'portrait':  {'guidance': 3.5, 'cn_scale': 0.7, 'cn_end': 0.8, 'canny_low': 50, 'canny_high': 200},
}
DEFAULT_STEPS = 28
DEFAULT_SEED  = 42
DEFAULT_MODE  = 'Reimagine'   # Imagine | Reimagine | Scene | Portrait

# ── Output directory ────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Default mode : {DEFAULT_MODE}')
print(f'LoRA source  : {LORA_SOURCE}')
print(f'Output dir   : {OUTPUT_DIR}')


## 7 — Load Models

`load_pipeline(mode)` handles VRAM switching — unloads the current pipeline before loading the new one.
The initial load uses `DEFAULT_MODE` set in cell-config.

⏱️ First run downloads ~24–29 GB depending on mode. Warm cache: ~1 minute.

In [ ]:
active_mode     = None
pipe            = None
controlnet      = None
depth_estimator = None


def _peft_patch():
    import peft.import_utils as _peft_utils
    import peft.tuners.lora.torchao as _peft_torchao
    _orig = _peft_utils.is_torchao_available
    def _safe():
        try: return _orig()
        except ImportError: return False
    _peft_utils.is_torchao_available = _safe
    _peft_torchao.is_torchao_available = _safe


def load_pipeline(mode: str) -> None:
    """Load the pipeline for the given mode, unloading the current one first."""
    global pipe, controlnet, depth_estimator, active_mode

    mode = mode.lower()
    if mode == active_mode:
        print(f'✓ {mode} already loaded.')
        return

    # ── Unload ───────────────────────────────────────────────────────────────────
    if active_mode:
        print(f'Unloading {active_mode} pipeline...')
    if pipe is not None:
        del pipe
        pipe = None
    if controlnet is not None:
        del controlnet
        controlnet = None
    if depth_estimator is not None:
        del depth_estimator
        depth_estimator = None
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total')

    # ── Load ────────────────────────────────────────────────────────────────────
    # Imagine uses plain FluxPipeline (text-to-image, no ControlNet, ~18–19 GB VRAM).
    if mode == 'imagine':
        print('Loading FLUX.1-dev (text-to-image)...')
        pipe = FluxPipeline.from_pretrained(
            BASE_MODEL_FLUX, torch_dtype=torch.bfloat16
        ).to('cuda')

    elif mode == 'reimagine':
        print('Loading FLUX.1-Kontext-dev...')
        pipe = FluxKontextPipeline.from_pretrained(
            BASE_MODEL_KONTEXT, torch_dtype=torch.bfloat16
        ).to('cuda')

    elif mode in ('scene', 'portrait'):
        print('Loading ControlNet...')
        controlnet = FluxControlNetModel.from_pretrained(
            CONTROLNET_MODEL, torch_dtype=torch.bfloat16
        )
        print('Loading FLUX.1-dev...')
        pipe = FluxControlNetPipeline.from_pretrained(
            BASE_MODEL_FLUX, controlnet=controlnet, torch_dtype=torch.bfloat16
        ).to('cuda')
        if mode == 'scene':
            print('Loading depth estimator (CPU)...')
            depth_estimator = hf_pipeline(
                'depth-estimation', model=DEPTH_MODEL, device='cpu'
            )
    else:
        raise ValueError(f'Unknown mode: {mode!r}')

    # ── LoRA ──────────────────────────────────────────────────────────────────────
    _peft_patch()
    # Gado Cartoon LoRA ── named adapter so cartoonify() can switch between styles
    if LORA_SOURCE == 'huggingface':
        pipe.load_lora_weights(LORA_HF_REPO, weight_name=LORA_HF_WEIGHT,
                               adapter_name=LORA_GADO_ADAPTER_NAME)
    elif os.path.isfile(LORA_DRIVE_PATH):
        pipe.load_lora_weights(os.path.dirname(LORA_DRIVE_PATH),
                               weight_name=os.path.basename(LORA_DRIVE_PATH),
                               adapter_name=LORA_GADO_ADAPTER_NAME)
    else:
        print(f'⚠️  Gado Cartoon LoRA not found: {LORA_DRIVE_PATH}')

    # Satirefic LoRA ── load only if the .safetensors file exists on Drive
    _sat_folder = os.path.dirname(LORA_SATIREFIC_PATH)
    _sat_file   = os.path.basename(LORA_SATIREFIC_PATH)
    _sat_exists = os.path.isfile(LORA_SATIREFIC_PATH)
    print(f'Satirefic LoRA path   : {LORA_SATIREFIC_PATH}')
    print(f'Satirefic path exists : {_sat_exists}')
    print(f'Satirefic folder      : {_sat_folder}')
    print(f'Satirefic file        : {_sat_file}')
    print(f'Satirefic adapter     : {LORA_SATIREFIC_ADAPTER_NAME}')
    if _sat_exists:
        pipe.load_lora_weights(_sat_folder, weight_name=_sat_file,
                               adapter_name=LORA_SATIREFIC_ADAPTER_NAME)
        print('✅ Satirefic LoRA loaded successfully')
    else:
        print(f'⚠️  Satirefic LoRA not found: {LORA_SATIREFIC_PATH}')

    # Disable all adapters on load; cartoonify() activates the user-selected one.
    pipe.disable_lora()

    active_mode = mode
    print(f'✅ {mode} pipeline ready.')


# ── Initial load ──────────────────────────────────────────────────────────────
load_pipeline(DEFAULT_MODE.lower())


## 8 — Story-to-Prompt and Wild Settings (Gemini)

Two Gemini functions:
- `build_prompt_from_story()` — Default mode: returns structured prompt only
- `build_wild_settings()` — Wild mode: returns prompt + parameter suggestions optimised for satirical impact

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DEFAULT MODE — prompt only
# ─────────────────────────────────────────────────────────────────────────────

GEMINI_SYSTEM_PROMPT = """You are a prompt engineer for a FLUX.1 LoRA fine-tuned on editorial and political cartoons.
Convert the user's story into a structured image generation prompt.

OUTPUT FORMAT — one line only, no explanation, no preamble:
gdo_cartoon <medium>, <technique>, <color>, <mood>, <commentary>, <composition>

RULES:
- Always start with: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration
- Always include: pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight
- Default color: black and white | monochrome | print cartoon | white background
  (use spot color red only if violence or urgency is central to the story)
- Derive mood, commentary, and composition from the story
- Pipe-separated keywords within each layer; comma between layers
- Max 5 keywords per layer
- Composition: estimate figure count, suggest layout, add speech bubble hint if there is dialogue

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | dark humour | critical | bleak, political commentary | scathing | power critique | deadpan | accusatory, two-group layout | standing figure left | queue of figures right | speech bubble top | eye level | wide shot

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, absurdist | dark | pompous | grotesque | confrontational, political condemnation | war metaphor | accusatory | ironic | scathing, large figure dominates upper frame | tiny figures lower third | desk as stage | top-heavy composition | eye level

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Prompt: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | duplicitous | tense | darkly comic, political commentary | diplomatic hypocrisy | sharp | deadpan | allegorical, two figure layout | shadow duality | frontal handshake foreground | fighting silhouettes background | centred composition | eye level"""


def build_prompt_from_story(story: str, trigger_word: str) -> str:
    if not story.strip():
        raise ValueError('Empty story.')
    if not GOOGLE_API_KEY:
        raise ValueError('GOOGLE_API_KEY not set in Colab Secrets.')

    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=GEMINI_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=300,
        ),
    )
    prompt = response.text.strip()

    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'gdo_cartoon':
        prompt = prompt.replace('gdo_cartoon', active_trigger, 1)

    return prompt


# ─────────────────────────────────────────────────────────────────────────────
# WILD MODE — prompt + parameter suggestions for maximum satirical impact
# ─────────────────────────────────────────────────────────────────────────────

WILD_SYSTEM_PROMPT = """You are a prompt engineer AND generation parameter optimizer for a FLUX.1 LoRA fine-tuned on satirical editorial cartoons.
Your goal: craft a structured prompt AND suggest generation parameters that together produce the sharpest, most satirically impactful cartoon possible.

OUTPUT FORMAT — exactly two lines, nothing else:
Line 1 (prompt): gdo_cartoon <medium>, <technique>, <color>, <mood>, <commentary>, <composition>
Line 2 (settings): {"mode": "reimagine|scene|portrait", "guidance": <2.0-6.0>, "steps": <28-40>, "cn_scale": <0.3-1.2>, "cn_end": <0.50-1.0>, "canny_low": <10-100>, "canny_high": <80-400>, "rationale": "<one sentence>"}

PROMPT RULES (same as Default):
- Start: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration
- Always include: pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight
- Default color: black and white | monochrome | print cartoon | white background
- Pipe-separated within layer, comma between layers, max 5 keywords per layer

MODE SELECTION:
- reimagine: total recomposition wanted, identity less critical, surreal/allegorical staging
- scene: power hierarchy, crowd, spatial layout must be preserved
- portrait: specific face must be immediately recognisable — caricature target

PARAMETER RULES FOR SATIRICAL IMPACT:
- Confrontational / scathing / dark → guidance 4.5–5.5, steps 35–40
- Whimsical / absurdist / ironic → guidance 2.5–3.5, steps 28–32
- Face identity critical → portrait, canny_low 20–40, canny_high 100–160, cn_scale 0.85–1.0
- Crowd / hierarchy scene → scene, cn_scale 0.85–0.95, steps 34–38
- Total recomposition / allegory → reimagine, guidance 2.5–3.0
- cn_end: 0.70 for more FLUX finishing freedom (richer hand-drawn feel), 0.80 standard, 0.85+ strict edge lock
- More complex compositions → more steps (36–40); single figure → 28–32 is sufficient

EXAMPLES:

Story: A politician hands out empty promises while people queue for food.
Line 1: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | dark humour | critical | bleak, political commentary | scathing | power critique | deadpan | accusatory, two-group layout | standing figure left | queue of figures right | speech bubble top | eye level | wide shot
Line 2: {"mode": "scene", "guidance": 4.5, "steps": 36, "cn_scale": 0.85, "cn_end": 0.75, "canny_low": 50, "canny_high": 200, "rationale": "Crowd-heavy power imbalance — depth map locks two-group layout; high guidance drives scathing political tone"}

Story: A general sits behind a massive desk covered in medals while tiny soldiers march across a map below him.
Line 1: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, absurdist | dark | pompous | grotesque | confrontational, political condemnation | war metaphor | accusatory | ironic | scathing, large figure dominates upper frame | tiny figures lower third | desk as stage | top-heavy composition | eye level
Line 2: {"mode": "portrait", "guidance": 5.0, "steps": 38, "cn_scale": 0.90, "cn_end": 0.70, "canny_low": 28, "canny_high": 130, "rationale": "General's pomposity demands tight face outlines; high guidance and steps maximise grotesque caricature detail"}

Story: Two leaders shake hands in front of cameras while their shadows fight each other behind them.
Line 1: gdo_cartoon editorial cartoon | political cartoon | caricature | newspaper illustration, pen and ink | cross-hatching | bold outlines | hand-drawn linework | varied line weight, black and white | monochrome | print cartoon | white background, satirical | ironic | duplicitous | tense | darkly comic, political commentary | diplomatic hypocrisy | sharp | deadpan | allegorical, two figure layout | shadow duality | frontal handshake foreground | fighting silhouettes background | centred composition | eye level
Line 2: {"mode": "reimagine", "guidance": 3.0, "steps": 32, "cn_scale": 0.70, "cn_end": 0.80, "canny_low": 50, "canny_high": 200, "rationale": "Shadow duality allegory needs Kontext's semantic recomposition — standard guidance lets FLUX interpret the staging freely"}"""


def build_wild_settings(story: str, trigger_word: str) -> tuple:
    """Returns (prompt_str, settings_dict) for Wild mode.
    settings_dict keys: mode, guidance, steps, cn_scale, cn_end, canny_low, canny_high, rationale.
    Returns (DEFAULT_PROMPT, {}) on any failure.
    """
    if not story.strip() or not GOOGLE_API_KEY:
        return DEFAULT_PROMPT, {}

    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.generate_content(
        model=GOOGLE_MODEL,
        contents=f'Story: {story.strip()}',
        config=genai_types.GenerateContentConfig(
            system_instruction=WILD_SYSTEM_PROMPT,
            temperature=0.7,
            max_output_tokens=500,
        ),
    )

    lines = [l.strip() for l in response.text.strip().split('\n') if l.strip()]
    prompt = lines[0] if lines else DEFAULT_PROMPT

    active_trigger = trigger_word.strip()
    if active_trigger and active_trigger != 'gdo_cartoon':
        prompt = prompt.replace('gdo_cartoon', active_trigger, 1)

    settings = {}
    for line in lines[1:]:
        if line.startswith('{'):
            try:
                settings = json.loads(line)
            except json.JSONDecodeError:
                pass
            break

    return prompt, settings


print('✓ Default build_prompt_from_story ready')
print('✓ Wild build_wild_settings ready')

## 9 — Inference

In [ ]:
# ── Status HTML helpers ───────────────────────────────────────────────────────
def _status(msg: str, state: str = 'processing') -> str:
    colors = {'idle': '#3f3f46', 'processing': '#a78bfa', 'done': '#34d399', 'warn': '#fbbf24'}
    c    = colors.get(state, colors['processing'])
    anim = ' animation:dot-pulse 1.2s ease-in-out infinite;' if state == 'processing' else ''
    return (
        f'<div class="cfy-status">'
        f'<div class="cfy-dot" style="background:{c};{anim}"></div>'
        f'<span>{msg}</span>'
        f'</div>'
    )

def _progress_html(step: int, total: int) -> str:
    pct = int(100 * step / max(total, 1))
    return (
        f'<div class="cfy-status">'
        f'<div class="cfy-dot" style="background:#a78bfa;animation:dot-pulse 1.2s ease-in-out infinite;"></div>'
        f'<div class="cfy-progress-wrap">'
        f'<div class="cfy-progress-label"><span>FLUX rendering</span>'
        f'<span style="color:#a78bfa;font-weight:700">{step} / {total}</span></div>'
        f'<div class="cfy-progress-track">'
        f'<div class="cfy-progress-fill" style="width:{pct}%"></div>'
        f'</div></div></div>'
    )

_STATUS_IDLE = _status('Upload a photo, describe your story, then hit Cartoonify.', 'idle')


def extract_canny(pil_image: Image.Image, low: int, high: int) -> Image.Image:
    img_np = np.array(pil_image)
    edges  = cv2.Canny(img_np, low, high)
    edges  = edges[:, :, None]
    edges  = np.concatenate([edges, edges, edges], axis=2)
    return Image.fromarray(edges)


def build_final_prompt(prompt: str, lora_style: str, trigger_word: str) -> str:
    """Assemble the final prompt sent to the image model.

    Always prepends GLOBAL_PROMPT_PREFIX and the LoRA trigger word.
    Appends SATIREFIC_STYLE_BOOST when Satirefic LoRA is selected.
    Neither prefix nor boost is duplicated if already present.
    """
    base = prompt.strip()

    # Prepend LoRA trigger word (existing behavior)
    trigger = trigger_word.strip()
    if trigger and not base.startswith(trigger):
        base = f'{trigger} {base}'

    # Global prompt prefix applied to every generation
    if not base.startswith(GLOBAL_PROMPT_PREFIX):
        base = f'{GLOBAL_PROMPT_PREFIX}, {base}'

    # Satirefic hidden style boost applied only when Satirefic LoRA is selected
    if lora_style == 'Satirefic LoRA' and SATIREFIC_STYLE_BOOST not in base:
        base = f'{base}, {SATIREFIC_STYLE_BOOST}'

    return base


def cartoonify(
    story,
    image,
    mode_label,
    wild_mode,        # bool  — Checkbox value
    lora_style,       # str   — 'None' | 'Gado Cartoon' | 'Satirefic LoRA'
    lora_strength,    # float — adapter weight (0.0–1.5)
    trigger_word,
    guidance_scale,
    num_steps,
    cn_scale,
    cn_end,
    canny_low,
    canny_high,
    seed,
    prompt_override,
):
    """Generator. Yields (canvas, status_html, result_state, view_toggle, guidance, steps, cn_scale, cn_end, canny_low, canny_high)."""
    if image is None and mode_label.lower() != 'imagine':
        raise gr.Error('Upload a photo first.')

    mode = mode_label.lower()

    # Auto-select trigger word from the chosen LoRA style (overrides manual input)
    style_cfg = LORA_STYLES.get(lora_style, {'adapter': None, 'trigger': ''})
    if style_cfg.get('trigger'):
        trigger_word = style_cfg['trigger']

    def emit(canvas=None, status=None, result=None, show_toggle=None,
             g=None, s=None, cn_s=None, cn_e=None, clow=None, chigh=None):
        if show_toggle is None:
            toggle_upd = gr.update()
        elif show_toggle:
            toggle_upd = gr.update(visible=True, value='Cartoon')
        else:
            toggle_upd = gr.update(visible=False)
        return (
            gr.update() if canvas is None else gr.update(value=canvas),
            gr.update() if status is None else gr.update(value=status),
            gr.update() if result is None else result,
            toggle_upd,
            gr.update() if g     is None else gr.update(value=g),
            gr.update() if s     is None else gr.update(value=s),
            gr.update() if cn_s  is None else gr.update(value=cn_s),
            gr.update() if cn_e  is None else gr.update(value=cn_e),
            gr.update() if clow  is None else gr.update(value=clow),
            gr.update() if chigh is None else gr.update(value=chigh),
        )

    def flux_with_progress(pipeline_call, total_steps):
        """Run pipeline in background thread; yield HTML progress strings then the PIL Image."""
        step_q     = queue.Queue()
        result_box = [None]
        error_box  = [None]

        def _cb(_, step, _ts, kwargs):
            step_q.put(step + 1)
            return kwargs

        def _run():
            with torch.inference_mode():
                try:
                    result_box[0] = pipeline_call(callback_on_step_end=_cb)
                except Exception as exc:
                    error_box[0] = exc

        t = threading.Thread(target=_run, daemon=True)
        t.start()

        while t.is_alive():
            try:
                step = step_q.get(timeout=0.3)
                yield _progress_html(step, total_steps)
            except queue.Empty:
                pass

        t.join()
        if error_box[0]:
            raise error_box[0]
        yield result_box[0]  # final item is the PIL Image

    # Working parameter values
    g_val     = guidance_scale
    s_val     = int(num_steps)
    cn_s_val  = cn_scale
    cn_e_val  = cn_end
    clow_val  = int(canny_low)
    chigh_val = int(canny_high)

    # ── Step 1: Prompt ────────────────────────────────────────────────────────
    if prompt_override.strip():
        prompt = prompt_override.strip()
        yield emit(status=_status('✓ Using manual prompt override', 'done'))

    elif wild_mode and story.strip():
        yield emit(status=_status('⚡ Wild — Gemini building prompt and tuning for satire…'))
        try:
            prompt, wild = build_wild_settings(story, trigger_word)

            if wild:
                g_val     = float(wild.get('guidance',  g_val))
                s_val     = int(wild.get('steps',       s_val))
                cn_s_val  = float(wild.get('cn_scale',  cn_s_val))
                cn_e_val  = float(wild.get('cn_end',    cn_e_val))
                clow_val  = int(wild.get('canny_low',   clow_val))
                chigh_val = int(wild.get('canny_high',  chigh_val))
                rationale = wild.get('rationale', '')

                mode_map  = {'imagine': 'Imagine', 'reimagine': 'Reimagine', 'scene': 'Scene', 'portrait': 'Portrait'}
                suggested = wild.get('mode', '').lower()
                mode_note = ''
                if suggested and suggested != mode:
                    mode_note = f' · Wild suggests {mode_map.get(suggested, suggested)} for this story'

                yield emit(
                    status=_status(f'✓ Wild applied — {rationale}{mode_note}', 'done'),
                    g=g_val, s=s_val,
                    cn_s=cn_s_val, cn_e=cn_e_val,
                    clow=clow_val, chigh=chigh_val,
                )
            else:
                yield emit(status=_status('✓ Prompt ready (settings parse failed — using current sliders)', 'done'))

        except Exception as exc:
            yield emit(status=_status(f'⚠️  Wild failed ({exc}) — falling back to Default', 'warn'))
            try:
                prompt = build_prompt_from_story(story, trigger_word)
            except Exception:
                prompt = DEFAULT_PROMPT

    elif story.strip():
        yield emit(status=_status('Gemini building your prompt…'))
        try:
            prompt = build_prompt_from_story(story, trigger_word)
            yield emit(status=_status('✓ Prompt ready', 'done'))
        except Exception as exc:
            yield emit(status=_status(f'⚠️  Gemini failed ({exc}) — using default prompt', 'warn'))
            prompt = DEFAULT_PROMPT

    else:
        prompt = DEFAULT_PROMPT
        yield emit(status=_status('No story — using default prompt', 'idle'))

    full_prompt = build_final_prompt(prompt, lora_style, trigger_word)

    # ── Step 2: Pipeline switch ───────────────────────────────────────────────
    if mode != active_mode:
        yield emit(status=_status(f'Switching to {mode_label} pipeline (~60–90 s)…'))
        load_pipeline(mode)
        yield emit(status=_status(f'✓ {mode_label} pipeline ready', 'done'))

    # ── LoRA adapter activation ────────────────────────────────────────────────
    active_adapter = style_cfg.get('adapter')
    if active_adapter:
        try:
            pipe.set_adapters([active_adapter], adapter_weights=[float(lora_strength)])
            pipe.enable_lora()
        except Exception as exc:
            yield emit(status=_status(
                f'⚠️  LoRA "{lora_style}" unavailable ({exc}) — running without LoRA', 'warn'))
            pipe.disable_lora()
    else:
        pipe.disable_lora()

    # ── Step 3: Resize ──────────────────────────────────────────────────────── (skipped for Imagine — text-to-image needs no input image)
    if mode != 'imagine':
        src = Image.fromarray(image).convert('RGB')
        src = src.resize((1024, 1024), Image.LANCZOS)
    generator = torch.Generator(device='cuda').manual_seed(int(seed))

    # ── Step 4: Inference ────────────────────────────────────────────────────
    result = None

    if mode == 'imagine':
        yield emit(status=_status('FLUX text-to-image rendering…'))
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                prompt=full_prompt,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                generator=generator,
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    elif mode == 'reimagine':
        yield emit(status=_status('FLUX Kontext rendering…'))
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                image=src,
                prompt=full_prompt,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    elif mode == 'scene':
        yield emit(status=_status('Reading scene depth…'))
        depth_out  = depth_estimator(src)
        depth_arr  = np.array(depth_out['depth'])
        d_min, d_max = depth_arr.min(), depth_arr.max()
        if d_max > d_min:
            depth_norm = ((depth_arr - d_min) / (d_max - d_min) * 255).astype(np.uint8)
        else:
            depth_norm = np.zeros_like(depth_arr, dtype=np.uint8)
        control_image = Image.fromarray(depth_norm).convert('RGB')
        yield emit(status=_status('✓ Depth map ready', 'done'))
        yield emit(status=_status('FLUX rendering…'))
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                prompt=full_prompt,
                control_image=control_image,
                control_mode=2,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                controlnet_conditioning_scale=cn_s_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    elif mode == 'portrait':
        yield emit(status=_status('Extracting portrait outlines…'))
        canny_image = extract_canny(src, clow_val, chigh_val)
        yield emit(status=_status('✓ Outlines extracted', 'done'))
        yield emit(status=_status('FLUX rendering…'))
        for item in flux_with_progress(
            lambda callback_on_step_end: pipe(
                prompt=full_prompt,
                control_image=canny_image,
                control_mode=0,
                width=1024, height=1024,
                num_inference_steps=s_val,
                guidance_scale=g_val,
                controlnet_conditioning_scale=cn_s_val,
                control_guidance_end=cn_e_val,
                generator=generator,
                joint_attention_kwargs={'scale': 1.0},
                max_sequence_length=512,
                callback_on_step_end=callback_on_step_end,
            ).images[0],
            s_val,
        ):
            if isinstance(item, str):
                yield emit(status=item)
            else:
                result = item

    if result is None:
        raise gr.Error('Generation produced no image — check the activity log above.')

    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    suffix = {'imagine': 'txt2img', 'reimagine': 'kontext', 'scene': 'depth', 'portrait': 'canny'}[mode]
    result.save(f'{OUTPUT_DIR}/{ts}_cartoonify_{suffix}.png')

    gc.collect()
    torch.cuda.empty_cache()

    yield emit(
        canvas=result,
        status=_status('✓ Done — saved to Drive', 'done'),
        result=result,
        show_toggle=True,
    )


print('✓ Inference functions ready')


## 10 — Launch Interface

Re-run this cell to restart the UI without reloading models.

**Layout:** Fixed left sidebar (mode, Wild toggle, fine-tune) + right canvas area (animated placeholder → uploaded photo → cartoon result).

**Wild mode (⚡ toggle):** Gemini reads the story and suggests parameters optimised for maximum satirical impact; sliders update to reflect what's being used.

**Original / Cartoon toggle:** Appears after generation completes — click to compare your uploaded photo against the cartoon result.


In [ ]:
MODE_DESCRIPTIONS = {
    'Imagine': (
        'FLUX generates from text only — no input photo needed. '
        'The story and prompt define the full composition. '
        'Best for new scenes with no reference image.'
    ),
    'Reimagine': (
        'FLUX recomposes the full image freely. '
        'The scene, staging, background and framing can all shift. '
        'Best for bold creative reinterpretation where you want maximum satirical transformation.'
    ),
    'Scene': (
        'A depth map locks the spatial layout and figure positions. '
        'Foreground and background relationships stay intact. '
        'Best for crowd scenes, protests, or architecture where structure must be preserved.'
    ),
    'Portrait': (
        'Canny edge detection traces face and body outlines before rendering. '
        'The person stays immediately recognisable in the cartoon. '
        'Best when satirising a specific individual — politician, CEO, public figure.'
    ),
}


def update_mode(mode):
    d             = DEFAULTS[mode.lower()]
    no_controlnet = mode in ('Imagine', 'Reimagine')  # neither uses ControlNet sliders
    is_portrait   = (mode == 'Portrait')
    return (
        MODE_DESCRIPTIONS[mode],
        gr.update(value=d['guidance']),
        gr.update(visible=not no_controlnet, value=d['cn_scale']),
        gr.update(visible=is_portrait,       value=d['cn_end']),
        gr.update(visible=is_portrait,       value=d['canny_low']),
        gr.update(visible=is_portrait,       value=d['canny_high']),
    )


def on_image_upload(image):
    if image is None:
        return (gr.update(visible=True), gr.update(value=None, visible=False),
                None, gr.update(visible=False), None)
    pil = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    return (gr.update(visible=False), gr.update(value=pil, visible=True),
            pil, gr.update(visible=False), None)


def toggle_view(choice, original, result):
    img = original if choice == 'Original' else result
    return gr.update(value=img)


CANVAS_PLACEHOLDER = '''
<div class="cfy-empty">
    <div class="cfy-empty-icon">&#9998;</div>
    <div class="cfy-ticker-outer">
        <div class="cfy-ticker-inner">
            <div class="cfy-quote">
                <p>&ldquo;Satire is the weapon of the powerless against the powerful.&rdquo;</p>
                <span>&mdash; Molly Ivins</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Comedy is simply a funny way of being serious.&rdquo;</p>
                <span>&mdash; Peter Ustinov</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Laughter is the shortest distance between two people.&rdquo;</p>
                <span>&mdash; Victor Borge</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;A caricature puts the face of a joke on the body of a truth.&rdquo;</p>
                <span>&mdash; Joseph Conrad</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;The purpose of satire is to strip away the veneer of comforting illusion.&rdquo;</p>
                <span>&mdash; Michael Foot</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Through laughter we hold up a mirror to the absurd.&rdquo;</p>
                <span>&mdash; Cartoonify</span>
            </div>
            <div class="cfy-quote">
                <p>&ldquo;Satire is the weapon of the powerless against the powerful.&rdquo;</p>
                <span>&mdash; Molly Ivins</span>
            </div>
        </div>
    </div>
    <p class="cfy-empty-hint">Upload a photo below to begin</p>
</div>
'''

CSS = '''
/* ── Tokens ──────────────────────────────────────────────── */
:root {
    --bg:         #0d0d0f;
    --surface:    #17171a;
    --surface-2:  #1e1e22;
    --border:     #2a2a30;
    --accent:     #a78bfa;
    --accent-dim: rgba(167,139,250,0.12);
    --text:       #f4f4f5;
    --muted:      #71717a;
    --amber:      #fbbf24;
    --amber-dim:  rgba(251,191,36,0.10);
    --radius:     12px;
}

/* ── Page reset ──────────────────────────────────────────── */
.gradio-container {
    font-family: "Inter", -apple-system, BlinkMacSystemFont, sans-serif !important;
    background: var(--bg) !important;
    max-width: 100% !important;
    margin: 0 !important; padding: 0 !important;
    color: var(--text) !important;
}
.gradio-container * { box-sizing: border-box; }
.dark, body, .main, .wrap, .contain, .block { background: transparent !important; }

/* ── Outer row (sidebar | canvas) ───────────────────────── */
#cfy-app { gap: 0 !important; }
#cfy-app > .wrap { gap: 0 !important; }

/* ── SIDEBAR ─────────────────────────────────────────────── */
#cfy-sidebar {
    min-width: 268px !important;
    max-width: 268px !important;
    background: var(--surface) !important;
    border-right: 1px solid var(--border) !important;
    padding: 1.25rem 1rem !important;
    overflow-y: auto !important;
    align-self: stretch !important;
}

.cfy-logo {
    display: block;
    font-size: 1.25rem; font-weight: 900; letter-spacing: -0.8px;
    color: var(--text); padding-bottom: 0.9rem;
    border-bottom: 1px solid var(--border); margin-bottom: 0.75rem;
}

.cfy-section-label {
    display: block;
    font-size: 0.62rem; font-weight: 800; text-transform: uppercase;
    letter-spacing: 0.1em; color: var(--muted);
    margin: 1rem 0 0.3rem;
}

/* Mode radio — vertical list */
#cfy-mode-nav { background: transparent !important; border: none !important; padding: 0 !important; }
#cfy-mode-nav .wrap { flex-direction: column !important; gap: 1px !important; }
#cfy-mode-nav label {
    padding: 0.45rem 0.6rem !important;
    border-radius: 7px !important; border: none !important;
    background: transparent !important;
    color: var(--muted) !important; font-weight: 500 !important;
    font-size: 0.85rem !important; cursor: pointer !important;
    transition: background 0.15s, color 0.15s !important;
}
#cfy-mode-nav label:has(input:checked) {
    background: var(--accent-dim) !important; color: var(--accent) !important;
}
#cfy-mode-nav label:hover:not(:has(input:checked)) {
    background: var(--surface-2) !important; color: var(--text) !important;
}
#cfy-mode-nav input[type="radio"] { display: none !important; }

/* Mode description */
#cfy-mode-desc { margin-top: 0.2rem !important; margin-bottom: 0.5rem !important; }
#cfy-mode-desc p {
    color: var(--muted) !important; font-size: 0.76rem !important;
    line-height: 1.55 !important; margin: 0 !important;
    padding: 0.25rem 0.65rem !important;
}

/* Wild toggle */
#cfy-wild {
    padding: 0.5rem 0.6rem !important; border-radius: 8px !important;
    background: var(--surface-2) !important; border: 1px solid var(--border) !important;
    transition: background 0.2s, border-color 0.2s !important;
}
#cfy-wild:has(input:checked) { background: var(--amber-dim) !important; border-color: rgba(251,191,36,0.3) !important; }
#cfy-wild label { color: var(--text) !important; font-weight: 600 !important; font-size: 0.79rem !important; cursor: pointer !important; gap: 0.5rem !important; }
#cfy-wild input[type="checkbox"] {
    appearance: none !important; -webkit-appearance: none !important;
    width: 2rem !important; height: 1rem !important;
    background: var(--border) !important; border-radius: 999px !important;
    position: relative !important; cursor: pointer !important; flex-shrink: 0 !important;
    transition: background 0.2s !important;
}
#cfy-wild input[type="checkbox"]:checked { background: var(--amber) !important; }
#cfy-wild input[type="checkbox"]::after {
    content: "" !important; position: absolute !important;
    top: 2px !important; left: 2px !important;
    width: 12px !important; height: 12px !important;
    background: white !important; border-radius: 50% !important;
    transition: transform 0.2s !important;
}
#cfy-wild input[type="checkbox"]:checked::after { transform: translateX(16px) !important; }

/* Accordion */
.gradio-container details {
    background: transparent !important; border: 1px solid var(--border) !important;
    border-radius: 8px !important; margin-top: 0.65rem !important;
}
.gradio-container details summary {
    color: var(--muted) !important; font-size: 0.78rem !important;
    font-weight: 600 !important; padding: 0.45rem 0.6rem !important; cursor: pointer !important;
}
.gradio-container details summary:hover { color: var(--text) !important; }
.gradio-container details[open] summary { color: var(--text) !important; border-bottom: 1px solid var(--border) !important; }
.gradio-container details .block { padding: 0.6rem !important; }

/* Sliders */
.gradio-container input[type="range"] { accent-color: var(--accent) !important; }
.gradio-container label span { color: var(--muted) !important; font-size: 0.76rem !important; }

/* ── CANVAS AREA ─────────────────────────────────────────── */
#cfy-canvas-area {
    padding: 1.25rem !important; min-width: 0 !important;
}
#cfy-canvas-area > .wrap { display: flex; flex-direction: column; gap: 0.65rem; }

/* Empty canvas placeholder */
#cfy-canvas-empty {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    min-height: 490px !important;
    display: flex !important; align-items: center !important; justify-content: center !important;
    padding: 0 !important;
}
.cfy-empty {
    display: flex; flex-direction: column; align-items: center;
    justify-content: center; gap: 1.75rem;
    padding: 2.5rem; width: 100%; height: 100%;
}
.cfy-empty-icon {
    font-size: 2.4rem; opacity: 0.18; user-select: none;
}

/* Ticker */
.cfy-ticker-outer {
    width: 100%; max-width: 560px; height: 5.5rem;
    overflow: hidden; position: relative;
    mask-image: linear-gradient(to bottom, transparent, black 25%, black 75%, transparent);
    -webkit-mask-image: linear-gradient(to bottom, transparent, black 25%, black 75%, transparent);
}
.cfy-ticker-inner { animation: ticker-scroll 28s linear infinite; }
.cfy-quote {
    height: 5.5rem;
    display: flex; flex-direction: column;
    align-items: center; justify-content: center;
    gap: 0.35rem; padding: 0 1rem; text-align: center;
}
.cfy-quote p {
    font-size: 0.9rem; color: var(--text); font-weight: 500; line-height: 1.5;
    margin: 0 !important; opacity: 0.75;
}
.cfy-quote span { font-size: 0.72rem; color: var(--muted); letter-spacing: 0.04em; }
@keyframes ticker-scroll {
    0%   { transform: translateY(0); }
    100% { transform: translateY(calc(-5.5rem * 6)); }
}
.cfy-empty-hint {
    font-size: 0.72rem; color: #3a3a42;
    letter-spacing: 0.06em; text-transform: uppercase; margin: 0;
}

/* Canvas image */
#cfy-canvas {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
}
#cfy-canvas img { border-radius: 10px !important; object-fit: contain !important; }

/* Original / Cartoon toggle — full-width segmented bar */
#cfy-toggle {
    width: 100% !important;
    background: var(--surface) !important; border: 1px solid var(--border) !important;
    border-radius: 10px !important; padding: 4px !important;
}
#cfy-toggle .wrap { display: flex !important; gap: 4px !important; width: 100% !important; }
#cfy-toggle label {
    flex: 1 !important; text-align: center !important;
    padding: 0.48rem 0 !important; border-radius: 8px !important;
    font-size: 0.82rem !important; font-weight: 600 !important;
    color: var(--muted) !important; cursor: pointer !important;
    transition: background 0.15s, color 0.15s !important;
    border: none !important; background: transparent !important;
}
#cfy-toggle label:has(input:checked) { background: var(--accent) !important; color: white !important; }
#cfy-toggle input[type="radio"] { display: none !important; }

/* Activity / status box */
#cfy-status-box {
    background: var(--surface-2) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    padding: 0.5rem 0.75rem !important;
    min-height: 2.4rem !important;
}
.cfy-status {
    display: flex; align-items: center; gap: 0.55rem;
    min-height: 1.4rem; padding: 0;
    font-family: "JetBrains Mono", "Fira Code", ui-monospace, monospace;
    font-size: 0.73rem; color: var(--muted);
}
.cfy-dot { width: 7px; height: 7px; border-radius: 50%; flex-shrink: 0; }
.cfy-progress-wrap { flex: 1; display: flex; flex-direction: column; gap: 3px; }
.cfy-progress-label { display: flex; justify-content: space-between; font-size: 0.73rem; color: var(--muted); }
.cfy-progress-track { height: 3px; background: var(--border); border-radius: 2px; overflow: hidden; }
.cfy-progress-fill { height: 100%; background: var(--accent); border-radius: 2px; transition: width 0.08s linear; }
@keyframes dot-pulse {
    0%, 100% { opacity: 1; transform: scale(1); }
    50%       { opacity: 0.3; transform: scale(0.7); }
}

/* Bottom input bar */
#cfy-input-row {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    padding: 0.75rem !important; gap: 0.75rem !important;
}
#cfy-input-row > .wrap { align-items: stretch !important; }

/* Text inputs */
.gradio-container textarea, .gradio-container input[type="text"],
.gradio-container input[type="number"] {
    background: var(--surface-2) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    color: var(--text) !important; font-size: 0.875rem !important;
}
.gradio-container textarea::placeholder,
.gradio-container input::placeholder { color: var(--muted) !important; }
.gradio-container textarea:focus, .gradio-container input:focus {
    border-color: var(--accent) !important; outline: none !important;
    box-shadow: 0 0 0 2px var(--accent-dim) !important;
}

/* Photo upload thumb — compact, no bleed */
.cfy-upload {
    border: 2px dashed var(--border) !important; border-radius: 8px !important;
    background: var(--surface-2) !important; transition: border-color 0.2s !important;
    overflow: hidden !important;
}
.cfy-upload:hover { border-color: var(--accent) !important; }
.cfy-upload .wrap { overflow: hidden !important; height: 100% !important; }
.cfy-upload p, .cfy-upload .upload-text, .cfy-upload span.or { display: none !important; }
.cfy-upload button.upload { display: none !important; }
.cfy-upload svg { width: 20px !important; height: 20px !important; opacity: 0.4 !important; }

/* Generate button */
#cfy-btn button {
    background: linear-gradient(135deg, #7c3aed 0%, #a78bfa 100%) !important;
    color: white !important; border: none !important; border-radius: 10px !important;
    font-size: 0.95rem !important; font-weight: 700 !important;
    height: 46px !important; width: 100% !important; letter-spacing: 0.02em !important;
    box-shadow: 0 4px 16px rgba(124,58,237,0.3) !important;
    transition: opacity 0.15s, transform 0.12s !important;
}
#cfy-btn button:hover  { opacity: 0.88 !important; transform: translateY(-1px) !important; }
#cfy-btn button:active { transform: translateY(0) !important; }
#cfy-btn button:disabled { opacity: 0.45 !important; cursor: wait !important; }

footer { display: none !important; }
'''


with gr.Blocks(
    css=CSS,
    theme=gr.themes.Base(
        primary_hue=gr.themes.colors.violet,
        neutral_hue=gr.themes.colors.zinc,
        font=gr.themes.GoogleFont('Inter'),
    ),
    title='Cartoonify',
) as demo:

    original_state = gr.State(None)
    result_state   = gr.State(None)

    with gr.Row(elem_id='cfy-app', equal_height=False):

        # ── LEFT SIDEBAR ──────────────────────────────────────────────────────
        with gr.Column(scale=2, min_width=268, elem_id='cfy-sidebar'):

            gr.HTML('<span class="cfy-logo">&#127912; Cartoonify</span>')

            gr.HTML('<span class="cfy-section-label">Mode</span>')
            mode_selector = gr.Radio(
                choices=['Imagine', 'Reimagine', 'Scene', 'Portrait'],
                value=DEFAULT_MODE, label='', show_label=False,
                elem_id='cfy-mode-nav',
            )
            mode_desc = gr.Markdown(
                value=MODE_DESCRIPTIONS[DEFAULT_MODE],
                elem_id='cfy-mode-desc',
            )

            gr.HTML('<span class="cfy-section-label">Options</span>')
            wild_toggle = gr.Checkbox(
                value=False,
                label='⚡ Wild — Gemini tunes for max satire',
                elem_id='cfy-wild',
            )

            gr.HTML('<span class="cfy-section-label">Style / LoRA</span>')
            # LoRA options: None = no adapter, Gado Cartoon = existing team LoRA,
            # Satirefic LoRA = personal trained LoRA at LORA_SATIREFIC_PATH
            lora_selector = gr.Radio(
                choices=['None', 'Gado Cartoon', 'Satirefic LoRA'],
                value='Gado Cartoon', label='', show_label=False,
                elem_id='cfy-lora-nav',
            )
            lora_strength_slider = gr.Slider(
                minimum=0.0, maximum=1.5, value=0.8, step=0.05,
                label='LoRA Strength',
            )

            with gr.Accordion('Fine-tune', open=False):
                guidance_slider = gr.Slider(
                    minimum=1.0, maximum=10.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['guidance'],
                    step=0.5, label='Guidance Scale',
                )
                steps_slider = gr.Slider(
                    minimum=10, maximum=50, value=DEFAULT_STEPS,
                    step=1, label='Inference Steps',
                )
                cn_scale_slider = gr.Slider(
                    minimum=0.1, maximum=1.5,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_scale'],
                    step=0.05, label='ControlNet Scale',
                    visible=DEFAULT_MODE != 'Reimagine',
                )
                cn_end_slider = gr.Slider(
                    minimum=0.3, maximum=1.0,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['cn_end'],
                    step=0.05, label='ControlNet Guidance End',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                canny_low_slider = gr.Slider(
                    minimum=0, maximum=200,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_low'],
                    step=10, label='Canny Low Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                canny_high_slider = gr.Slider(
                    minimum=50, maximum=500,
                    value=DEFAULTS[DEFAULT_MODE.lower()]['canny_high'],
                    step=10, label='Canny High Threshold',
                    visible=DEFAULT_MODE == 'Portrait',
                )
                seed_input    = gr.Number(value=DEFAULT_SEED, label='Seed', precision=0)
                trigger_input = gr.Textbox(value=DEFAULT_TRIGGER, label='LoRA Trigger Word', lines=1)
                with gr.Accordion('Edit prompt directly', open=False):
                    prompt_input = gr.Textbox(
                        label='Prompt override',
                        placeholder='Leave empty — Gemini builds the prompt from your story.',
                        lines=3, max_lines=6, value='',
                    )

        # ── CANVAS + INPUT ────────────────────────────────────────────────────
        with gr.Column(scale=7, elem_id='cfy-canvas-area'):

            # Animated quote ticker — hidden once an image is loaded
            canvas_placeholder = gr.HTML(
                value=CANVAS_PLACEHOLDER,
                elem_id='cfy-canvas-empty',
                visible=True,
            )

            # Main canvas — shows uploaded photo, then the cartoon result
            canvas_display = gr.Image(
                label='', type='pil', interactive=False,
                elem_id='cfy-canvas', height=490,
                visible=False,
            )

            # Original / Cartoon toggle — revealed after generation completes
            view_toggle = gr.Radio(
                choices=['Original', 'Cartoon'],
                value='Cartoon', label='', show_label=False,
                visible=False, elem_id='cfy-toggle',
            )

            gr.HTML('<span class="cfy-section-label" style="margin:0 0 0.2rem">Activity</span>')
            status_output = gr.HTML(value=_STATUS_IDLE, elem_id='cfy-status-box')

            # Bottom bar: story text + compact photo upload thumb
            with gr.Row(elem_id='cfy-input-row', equal_height=True):
                with gr.Column(scale=5):
                    story_input = gr.Textbox(
                        label='', lines=3, max_lines=6,
                        placeholder=(
                            'Describe your story or what to change…\n'
                            'e.g. A politician handing out empty promises while people queue for food'
                        ),
                    )
                with gr.Column(scale=2, min_width=130):
                    img_input = gr.Image(
                        label='Photo', type='numpy', height=104,
                        elem_classes=['cfy-upload'],
                        sources=['upload', 'clipboard'],
                    )

            generate_btn = gr.Button('Cartoonify', variant='primary', elem_id='cfy-btn')

    # ── Event wiring ──────────────────────────────────────────────────────────

    def on_lora_change(style):
        cfg = LORA_STYLES.get(style, {})
        return gr.update(value=cfg.get('trigger', ''))

    lora_selector.change(
        fn=on_lora_change,
        inputs=[lora_selector],
        outputs=[trigger_input],
    )

    mode_selector.change(
        fn=update_mode, inputs=[mode_selector],
        outputs=[mode_desc, guidance_slider, cn_scale_slider,
                 cn_end_slider, canny_low_slider, canny_high_slider],
    )

    img_input.change(
        fn=on_image_upload, inputs=[img_input],
        outputs=[canvas_placeholder, canvas_display, original_state, view_toggle, result_state],
    )

    generate_btn.click(
        fn=cartoonify,
        inputs=[
            story_input, img_input, mode_selector, wild_toggle,
            lora_selector, lora_strength_slider,
            trigger_input, guidance_slider, steps_slider,
            cn_scale_slider, cn_end_slider, canny_low_slider, canny_high_slider,
            seed_input, prompt_input,
        ],
        outputs=[
            canvas_display, status_output, result_state,
            view_toggle,
            guidance_slider, steps_slider,
            cn_scale_slider, cn_end_slider,
            canny_low_slider, canny_high_slider,
        ],
        api_name='cartoonify',
    )

    view_toggle.change(
        fn=toggle_view,
        inputs=[view_toggle, original_state, result_state],
        outputs=[canvas_display],
    )


demo.launch(share=True, show_error=True, quiet=False)
